# TASK 4 — Multilingual clinical conversation benchmark

This notebook extends the existing Qwen clinical extraction work into a **separate, reproducible comparison** for Arabic, Bengali, Malayalam, and Urdu. It is configured for the **Tesla T4 (15 GiB)** shown in your Colab screenshot. Models are loaded sequentially to limit GPU memory use.

## Important baseline correction

The uploaded fine-tuning notebook loads `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` and saves a LoRA adapter from that base. This notebook therefore treats **Qwen2.5-1.5B base vs its matching fine-tuned adapter** as the comparable Qwen baseline. The adapter cannot be attached to Qwen2.5-7B. A 7B model can be added as a separate 4-bit candidate, but it is not the same fine-tuned baseline.

## Evaluation design

1. Create a manifest with the same approved audio clips, verified transcripts, and verified reference reports for each language.
2. Compare Whisper `large-v3-turbo`, SeamlessM4T v2 Large, and (optionally) AI4Bharat IndicConformer on speech recognition.
3. Compare the clinical extraction models on the **same human-verified transcripts**. This isolates extraction quality from ASR errors.
4. Track WER/CER, script correctness, run time, peak VRAM, prompt/output tokens, and human-reviewed clinical field accuracy. ROUGE is supplemental and does not establish clinical correctness.

Use only synthetic, deidentified, or otherwise approved data. Do not put patient identifiers, access tokens, or API keys in the manifest. The benchmark does not provide medical advice; clinicians must review outputs.

## Step 1 — Set up Colab

Open a **copy** of this notebook in Colab, choose **Runtime → Change runtime type → T4 GPU**, then run each cell in order. The next cell installs the speech and evaluation dependencies. Run the install once per fresh runtime.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -U openai-whisper jiwer transformers accelerate bitsandbytes peft sentencepiece pandas

In [ ]:
import os, gc, json, time, math, re, unicodedata, warnings
from pathlib import Path
import pandas as pd
import torch

assert torch.cuda.is_available(), "Select a GPU runtime: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("Total GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

def gpu_peak_gib():
    return round(torch.cuda.max_memory_allocated() / 1024**3, 3)

def clear_gpu(*names):
    for name in names:
        globals().pop(name, None)
    gc.collect()
    torch.cuda.empty_cache()

## Step 2 — Add your samples

Create this Drive folder and a CSV named `benchmark_manifest.csv`. Put audio files under the same folder (or use absolute Drive paths). Include **at least 5–10 clips per language** for a more useful pilot; one clip per language only checks that the code runs.

Required columns:

| Column | Meaning |
|---|---|
| `sample_id` | Unique non-identifying ID, e.g. `ar_001` |
| `language` | One of `Arabic`, `Bengali`, `Malayalam`, `Urdu` |
| `audio_path` | Path to audio file, e.g. `/content/drive/MyDrive/multilingual_clinical_benchmark/audio/ar_001.wav` |
| `reference_transcript` | Human-verified transcript in the spoken language/script |
| `reference_report` | Human-verified structured report, in the target output language |

Keep test samples separate from any training/fine-tuning set. If reference reports are in English, set `OUTPUT_LANGUAGE` to `English` below; otherwise leave it as `same as input`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/multilingual_clinical_benchmark')
MANIFEST_PATH = PROJECT_DIR / 'benchmark_manifest.csv'
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Edit this if your Drive folder differs.
print('Expected manifest:', MANIFEST_PATH)

In [ ]:
LANGUAGES = {'Arabic', 'Bengali', 'Malayalam', 'Urdu'}
manifest = pd.read_csv(MANIFEST_PATH).fillna('')
required = {'sample_id','language','audio_path','reference_transcript','reference_report'}
missing = required - set(manifest.columns)
assert not missing, f'Missing manifest columns: {sorted(missing)}'
assert set(manifest.language).issubset(LANGUAGES), f'Use language values from: {sorted(LANGUAGES)}'
assert manifest.sample_id.is_unique, 'sample_id values must be unique'
assert manifest.audio_path.map(lambda p: Path(p).exists()).all(), 'At least one audio_path does not exist'
print(manifest.groupby('language').size().reindex(sorted(LANGUAGES), fill_value=0))
manifest.head(3)

## Step 3 — Speech recognition: Whisper `large-v3-turbo`

Whisper's multilingual transcription mode returns text in the spoken language. Here the language is specified because it is known from the balanced test set. Model weights are downloaded on first run. On the T4, this model is the main Whisper candidate; run it alone, save results, then clear it before loading SeamlessM4T.

In [ ]:
import whisper
from jiwer import wer, cer

WHISPER_LANG = {'Arabic':'Arabic','Bengali':'Bengali','Malayalam':'Malayalam','Urdu':'Urdu'}

torch.cuda.reset_peak_memory_stats()
whisper_model = whisper.load_model('large-v3-turbo', device='cuda')
whisper_rows = []
for row in manifest.to_dict('records'):
    t0 = time.perf_counter()
    pred = whisper_model.transcribe(
        row['audio_path'], language=WHISPER_LANG[row['language']], task='transcribe',
        fp16=True, temperature=0.0, verbose=False
    )['text'].strip()
    whisper_rows.append({**row, 'asr_model':'whisper-large-v3-turbo', 'asr_prediction':pred,
                         'asr_seconds':time.perf_counter()-t0})
whisper_df = pd.DataFrame(whisper_rows)
whisper_df.to_csv(RESULTS_DIR/'asr_whisper_turbo.csv', index=False)
print('Saved:', RESULTS_DIR/'asr_whisper_turbo.csv')
print('Peak allocated GPU GiB:', gpu_peak_gib())
whisper_df[['sample_id','language','asr_seconds','asr_prediction']].head()

## Step 4 — Speech recognition: SeamlessM4T v2 Large

This uses the dedicated speech-to-text class from Transformers. It performs ASR (transcription), not translation. Load it only after the Whisper cell has finished and released its model.

In [ ]:
import librosa
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText

SEAMLESS_ID = 'facebook/seamless-m4t-v2-large'
torch.cuda.reset_peak_memory_stats()
seamless_processor = AutoProcessor.from_pretrained(SEAMLESS_ID)
seamless_model = SeamlessM4Tv2ForSpeechToText.from_pretrained(
    SEAMLESS_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True
).to('cuda').eval()
seamless_rows = []
for row in manifest.to_dict('records'):
    audio, sr = librosa.load(row['audio_path'], sr=16000, mono=True)
    inputs = seamless_processor(audio, sampling_rate=16000, return_tensors='pt')
    inputs = {k:v.to('cuda') for k,v in inputs.items()}
    t0 = time.perf_counter()
    with torch.inference_mode():
        logits = seamless_model(**inputs).logits
        pred_ids = torch.argmax(logits, dim=-1)
    pred = seamless_processor.batch_decode(pred_ids, skip_special_tokens=True)[0].strip()
    seamless_rows.append({**row, 'asr_model':'seamless-m4t-v2-large', 'asr_prediction':pred,
                          'asr_seconds':time.perf_counter()-t0})
    del audio, inputs, logits, pred_ids
seamless_df = pd.DataFrame(seamless_rows)
seamless_df.to_csv(RESULTS_DIR/'asr_seamless_v2.csv', index=False)
print('Saved:', RESULTS_DIR/'asr_seamless_v2.csv')
print('Peak allocated GPU GiB:', gpu_peak_gib())
seamless_df[['sample_id','language','asr_seconds','asr_prediction']].head()

## Step 5 — Optional IndicConformer comparison

IndicConformer is a useful specialist comparison for Bengali, Malayalam, and Urdu. It does not cover Arabic. Its official repository uses an AI4Bharat NeMo branch, so installation is separate and can change the Colab environment. Run this in a **fresh Colab runtime**, following the official [AI4Bharat IndicConformerASR instructions](https://github.com/AI4Bharat/IndicConformerASR). Do not run the NeMo installation in the same session as the core benchmark unless needed. Save its output with the same `sample_id` values so it can be joined to these results.

## Step 6 — Compare ASR results

WER and CER are computed after Unicode normalization. CER is especially useful for these scripts. Review predictions manually too: low error scores do not ensure medical terms, dosage numbers, or script are correct.

In [ ]:
def normalize_text(s):
    s = unicodedata.normalize('NFC', str(s or '')).strip()
    s = re.sub(r'\s+', ' ', s)
    return s

def score_asr(df):
    rows=[]
    for _, r in df.iterrows():
        ref, hyp = normalize_text(r['reference_transcript']), normalize_text(r['asr_prediction'])
        rows.append({'sample_id':r['sample_id'],'language':r['language'],'asr_model':r['asr_model'],
                     'WER':wer(ref,hyp) if ref else None,'CER':cer(ref,hyp) if ref else None,
                     'seconds':r['asr_seconds']})
    return pd.DataFrame(rows)

asr_scores = pd.concat([score_asr(whisper_df), score_asr(seamless_df)], ignore_index=True)
asr_summary = asr_scores.groupby(['language','asr_model'],as_index=False).agg(
    samples=('sample_id','count'), mean_WER=('WER','mean'), mean_CER=('CER','mean'),
    mean_seconds=('seconds','mean'))
asr_scores.to_csv(RESULTS_DIR/'asr_per_sample_scores.csv',index=False)
asr_summary.to_csv(RESULTS_DIR/'asr_summary.csv',index=False)
asr_summary

## Step 7 — Clinical extraction model comparison

For fair comparison, this section uses each sample's **human-verified reference transcript** as input. That isolates extraction quality from ASR quality. The prompt requests compact JSON and explicitly tells the model not to infer missing details. The cell records prompt tokens, generated tokens, runtime, and peak allocated GPU memory.

Set the adapter path to your existing `final_adapter` folder if it is present in Drive. A 1.5B adapter must use the matching Qwen2.5-1.5B base. The adapter path may need adjustment to match where your original notebook saved it.

In [ ]:
# Change only if your files are stored elsewhere.
QWEN_BASE_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
QWEN_ADAPTER_DIR = '/content/drive/MyDrive/qwen_clinical_finetuned_3k/final_adapter'
OUTPUT_LANGUAGE = 'same as input'  # set to 'English' if that is your reference report language
MAX_INPUT_TOKENS = 3072
MAX_NEW_TOKENS = 384

# Candidates run sequentially; set False to skip a model.
RUN_FINETUNED_QWEN_15B = True
RUN_GEMMA3_4B = True
RUN_QWEN3_4B = True

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

def make_prompt(language, transcript):
    return f"""Extract only facts explicitly present in this clinical conversation. Do not diagnose beyond what is stated. Do not invent missing values. Use null for absent information. Preserve medication names, dosages, units, and numbers exactly. Return one compact JSON object with these keys: patient_information, chief_complaint, symptoms, history_of_present_illness, past_medical_history, family_history, examination_and_labs, diagnosis_as_stated, medications, plan, follow_up, warning_signs. Write the report in {OUTPUT_LANGUAGE} (input language: {language}).

Conversation:
{transcript}

JSON:"""

def load_quantized_candidate(model_id, adapter_dir=None):
    qconfig = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                 bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=qconfig, device_map='auto',
        torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True
    )
    if adapter_dir:
        if not Path(adapter_dir).exists():
            raise FileNotFoundError(f'LoRA adapter not found: {adapter_dir}')
        model = PeftModel.from_pretrained(model, adapter_dir)
    model.eval()
    return model, tokenizer

def run_extractor(model_id, label, adapter_dir=None):
    model, tok = load_quantized_candidate(model_id, adapter_dir)
    torch.cuda.reset_peak_memory_stats()
    rows=[]
    for row in manifest.to_dict('records'):
        prompt = make_prompt(row['language'], row['reference_transcript'])
        messages=[{'role':'user','content':prompt}]
        if getattr(tok, 'chat_template', None):
            prompt_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            prompt_text = prompt
        enc = tok(prompt_text, return_tensors='pt', truncation=True, max_length=MAX_INPUT_TOKENS)
        enc = {k:v.to(model.device) for k,v in enc.items()}
        prompt_tokens = int(enc['input_ids'].shape[-1])
        t0=time.perf_counter()
        with torch.inference_mode():
            out=model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                               pad_token_id=tok.eos_token_id)
        elapsed=time.perf_counter()-t0
        new_ids=out[0, prompt_tokens:]
        pred=tok.decode(new_ids, skip_special_tokens=True).strip()
        rows.append({'sample_id':row['sample_id'],'language':row['language'],'extractor_model':label,
                     'reference_transcript':row['reference_transcript'],
                     'reference_report':row['reference_report'],'predicted_report':pred,
                     'prompt_tokens':prompt_tokens,'output_tokens':int(new_ids.numel()),
                     'seconds':elapsed,'peak_gpu_gib':gpu_peak_gib()})
        del enc, out, new_ids
    del model, tok
    clear_gpu()
    return pd.DataFrame(rows)

In [ ]:
extract_frames=[]
if RUN_FINETUNED_QWEN_15B:
    extract_frames.append(run_extractor(QWEN_BASE_ID,'qwen2.5-1.5b-finetuned',QWEN_ADAPTER_DIR))
if RUN_GEMMA3_4B:
    extract_frames.append(run_extractor('google/gemma-3-4b-it','gemma-3-4b'))
if RUN_QWEN3_4B:
    extract_frames.append(run_extractor('Qwen/Qwen3-4B','qwen3-4b'))

extraction_results = pd.concat(extract_frames, ignore_index=True)
extraction_results.to_csv(RESULTS_DIR/'clinical_extraction_predictions.csv', index=False)
print('Saved:', RESULTS_DIR/'clinical_extraction_predictions.csv')
extraction_results[['sample_id','language','extractor_model','prompt_tokens','output_tokens','seconds','peak_gpu_gib']].head()

## Step 8 — Review accuracy and select models

A clinician or qualified reviewer should score each generated report against the reference. Use the CSV below to record **field-level correctness**, invented facts, and critical omissions. Do not use a single ROUGE score as clinical accuracy. For a defensible pilot, report per-language sample counts and results; expand the set before making strong claims.

In [ ]:
review = extraction_results[['sample_id','language','extractor_model','reference_report','predicted_report',
                              'prompt_tokens','output_tokens','seconds','peak_gpu_gib']].copy()
review['field_accuracy_0_to_1'] = ''
review['critical_error_yes_no'] = ''
review['hallucinated_fact_yes_no'] = ''
review['reviewer_notes'] = ''
review_path=RESULTS_DIR/'clinical_review_template.csv'
review.to_csv(review_path,index=False)
print('Review predictions and fill the three scoring columns:',review_path)

print('Results folder:', RESULTS_DIR)
print('Files:', sorted(p.name for p in RESULTS_DIR.glob('*')))

## Results interpretation checklist

- Compare **WER and CER per language**, not only one overall average.
- Inspect Arabic dialect and code-switching cases; Whisper's language label does not guarantee dialect accuracy.
- Check that Bengali, Malayalam, and Urdu outputs use the expected native script.
- For reports, compare each required field and separately flag dosage/number errors, omitted red flags, and invented diagnoses/medications.
- Compare token counts only among extractors using the same reference transcript and same prompt/schema.
- Record failed runs and GPU memory. Do not silently drop failed samples.
- Keep the baseline identification precise: this notebook's adapter is Qwen2.5-1.5B. Add Qwen2.5-7B only as another separately named model, without this adapter.

References: [OpenAI Whisper](https://github.com/openai/whisper), [SeamlessM4T v2 Transformers documentation](https://huggingface.co/docs/transformers/model_doc/seamless_m4t_v2), [AI4Bharat IndicConformerASR](https://github.com/AI4Bharat/IndicConformerASR).